In [1]:
import pandas as pd

kmeans = pd.read_csv(
    "../results/kmeans/evaluacion_kmeans.csv"
)

kmedoids = pd.read_csv(
    "../results/kmedoids/evaluacion_kmedoids.csv"
)

jerarquico = pd.read_csv(
    "../results/jerarquico/evaluacion_jerarquico.csv"
)

hdbscan = pd.read_csv(
    "../results/hdbscan/evaluacion_hdbscan.csv"
)


comparacion = pd.concat(
    [
        kmeans,
        kmedoids,
        jerarquico,
        hdbscan
    ],
    ignore_index=True
)


comparacion

,modelo,configuracion,silhouette,davies_bouldin,ruido,estudiantes_total,estudiantes_agrupados
0,K-Means,k=2,0.493255,0.898488,NaN,NaN,NaN
1,K-Medoids,k=2,0.492114,0.897678,NaN,NaN,NaN
2,Jerárquico,"ward, k=2",0.497297,0.815482,NaN,NaN,NaN
3,HDBSCAN,"min_cluster_size=30, min_samples=10",0.593727,0.598189,43.0,244.0,201.0


In [2]:

from pathlib import Path

total = len(pd.read_csv(
    "../data/processed/X_clustering.csv"
))

comparacion["estudiantes_total"] = total
comparacion["estudiantes_agrupados"] = total

es_hdbscan = comparacion["modelo"] == "HDBSCAN"

comparacion.loc[
    es_hdbscan, "estudiantes_agrupados"
] = int(hdbscan.iloc[0]["estudiantes_agrupados"])

comparacion["cobertura"] = (
    comparacion["estudiantes_agrupados"]
    / comparacion["estudiantes_total"]
)

comparacion["silhouette_ajustado"] = (
    comparacion["silhouette"]
    * comparacion["cobertura"]
)

Path("../results/comparacion").mkdir(
    parents=True, exist_ok=True
)

print("COMPARACIÓN CON COBERTURA:")
print(
    comparacion[[
        "modelo",
        "silhouette",
        "davies_bouldin",
        "estudiantes_agrupados",
        "estudiantes_total",
        "cobertura",
        "silhouette_ajustado"
    ]].round(6).to_string(index=False)
)


COMPARACIÓN CON COBERTURA:
    modelo  silhouette  davies_bouldin  estudiantes_agrupados  estudiantes_total  cobertura  silhouette_ajustado
   K-Means    0.493255        0.898488                    244                244    1.00000             0.493255
 K-Medoids    0.492114        0.897678                    244                244    1.00000             0.492114
Jerárquico    0.497297        0.815482                    244                244    1.00000             0.497297
   HDBSCAN    0.593727        0.598189                    201                244    0.82377             0.489095


In [3]:
comparacion.to_csv(
    "../results/comparacion/matriz_comparativa.csv",
    index=False,
    encoding="utf-8-sig"
)

In [4]:

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

trayectorias = pd.read_csv(
    "../data/processed/trayectorias_academicas.csv"
)

X = pd.read_csv(
    "../data/processed/X_clustering.csv"
)

assert len(trayectorias) == len(X)
assert trayectorias["id_estudiante"].is_unique

X_reconstruida = StandardScaler().fit_transform(
    trayectorias[X.columns.tolist()]
)

assert np.allclose(
    X.to_numpy(),
    X_reconstruida,
    rtol=1e-7,
    atol=1e-8
), "La matriz no coincide con las trayectorias"

archivos = {
    "K-Means": "../results/kmeans/asignaciones_kmeans.csv",
    "K-Medoids": "../results/kmedoids/asignaciones_kmedoids.csv",
    "Jerárquico": "../results/jerarquico/asignaciones_jerarquico.csv",
    "HDBSCAN": "../results/hdbscan/asignaciones_hdbscan.csv"
}

for nombre, ruta in archivos.items():
    asignaciones = pd.read_csv(ruta)

    assert len(asignaciones) == len(X), nombre
    assert np.array_equal(
        asignaciones["fila"].to_numpy(),
        np.arange(len(X))
    ), f"Orden de filas incorrecto: {nombre}"

    print(f"{nombre}: {len(asignaciones)} filas verificadas")

print("\nMatriz y posiciones verificadas correctamente")
print("Estudiantes:", len(X))


K-Means: 244 filas verificadas
K-Medoids: 244 filas verificadas
Jerárquico: 244 filas verificadas
HDBSCAN: 244 filas verificadas

Matriz y posiciones verificadas correctamente
Estudiantes: 244


In [5]:

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score
)

X = pd.read_csv("../data/processed/X_clustering.csv")

archivos = {
    "K-Means": (
        "../results/kmeans/asignaciones_kmeans.csv",
        "../results/kmeans/evaluacion_kmeans.csv"
    ),
    "K-Medoids": (
        "../results/kmedoids/asignaciones_kmedoids.csv",
        "../results/kmedoids/evaluacion_kmedoids.csv"
    ),
    "Jerárquico": (
        "../results/jerarquico/asignaciones_jerarquico.csv",
        "../results/jerarquico/evaluacion_jerarquico.csv"
    ),
    "HDBSCAN": (
        "../results/hdbscan/asignaciones_hdbscan.csv",
        "../results/hdbscan/evaluacion_hdbscan.csv"
    )
}

asignaciones = {}

for nombre, (ruta_asig, ruta_eval) in archivos.items():
    df_asig = pd.read_csv(ruta_asig)

    assert np.array_equal(
        df_asig["fila"].to_numpy(),
        np.arange(len(X))
    ), f"Filas incorrectas: {nombre}"

    etiquetas = df_asig["cluster"].to_numpy()
    mask_original = etiquetas != -1

    sil = silhouette_score(
        X.loc[mask_original],
        etiquetas[mask_original]
    )

    db = davies_bouldin_score(
        X.loc[mask_original],
        etiquetas[mask_original]
    )

    guardado = pd.read_csv(ruta_eval).iloc[0]

    assert np.isclose(
        sil, guardado["silhouette"], atol=1e-5
    ), f"Silhouette no coincide: {nombre}"

    assert np.isclose(
        db, guardado["davies_bouldin"], atol=1e-5
    ), f"Davies-Bouldin no coincide: {nombre}"

    asignaciones[nombre] = etiquetas
    print(f"{nombre}: métricas originales verificadas")

mascara_comun = asignaciones["HDBSCAN"] != -1
X_comun = X.loc[mascara_comun]

resultados = []

for nombre, etiquetas in asignaciones.items():
    etiquetas_comunes = etiquetas[mascara_comun]

    cantidad_grupos = len(np.unique(etiquetas_comunes))

    if 2 <= cantidad_grupos < len(X_comun):
        sil = silhouette_score(
            X_comun, etiquetas_comunes
        )
        db = davies_bouldin_score(
            X_comun, etiquetas_comunes
        )
    else:
        sil = np.nan
        db = np.nan

    resultados.append({
        "modelo": nombre,
        "estudiantes_evaluados": len(X_comun),
        "silhouette_comun": sil,
        "davies_bouldin_comun": db,
        "cobertura_total": (etiquetas != -1).mean()
    })

comparacion_comun = pd.DataFrame(resultados)

ruta_salida = Path("../results/comparacion")
ruta_salida.mkdir(parents=True, exist_ok=True)

comparacion_comun.to_csv(
    ruta_salida / "comparacion_muestra_comun.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nCOMPARACIÓN SOBRE LA MISMA POBLACIÓN:")
print(comparacion_comun.round(6).to_string(index=False))


K-Means: métricas originales verificadas
K-Medoids: métricas originales verificadas
Jerárquico: métricas originales verificadas
HDBSCAN: métricas originales verificadas

COMPARACIÓN SOBRE LA MISMA POBLACIÓN:
    modelo  estudiantes_evaluados  silhouette_comun  davies_bouldin_comun  cobertura_total
   K-Means                    201          0.586421              0.625052          1.00000
 K-Medoids                    201          0.581485              0.640508          1.00000
Jerárquico                    201          0.593727              0.598189          1.00000
   HDBSCAN                    201          0.593727              0.598189          0.82377


In [6]:

from sklearn.metrics import adjusted_rand_score

mascara = asignaciones["HDBSCAN"] != -1

etiquetas_ward = asignaciones["Jerárquico"][mascara]
etiquetas_hdbscan = asignaciones["HDBSCAN"][mascara]

tabla_cruce = pd.crosstab(
    pd.Series(etiquetas_ward, name="Ward"),
    pd.Series(etiquetas_hdbscan, name="HDBSCAN")
)

print("COMPARACIÓN DE AGRUPACIONES:")
print(tabla_cruce)

ari = adjusted_rand_score(
    etiquetas_ward,
    etiquetas_hdbscan
)

print("\nÍndice Rand ajustado (ARI):", round(ari, 6))

print("\nASIGNACIÓN DE LOS 43 CASOS DE RUIDO EN WARD:")
print(
    pd.Series(asignaciones["Jerárquico"][~mascara])
    .value_counts()
    .sort_index()
)


COMPARACIÓN DE AGRUPACIONES:
HDBSCAN   0    1
Ward            
0         0  166
1        35    0

Índice Rand ajustado (ARI): 1.0

ASIGNACIÓN DE LOS 43 CASOS DE RUIDO EN WARD:
0    23
1    20
Name: count, dtype: int64


In [7]:

tabla_final = comparacion_comun.merge(
    comparacion[[
        "modelo",
        "silhouette",
        "davies_bouldin",
        "estudiantes_agrupados"
    ]],
    on="modelo",
    validate="one_to_one"
)

tabla_final = tabla_final.rename(columns={
    "estudiantes_agrupados": "n_original",
    "silhouette": "silhouette_original",
    "davies_bouldin": "davies_original",
    "estudiantes_evaluados": "n_comun",
    "davies_bouldin_comun": "davies_comun"
})

tabla_final = tabla_final[[
    "modelo",
    "n_original",
    "silhouette_original",
    "davies_original",
    "n_comun",
    "silhouette_comun",
    "davies_comun",
    "cobertura_total"
]]

tabla_final.to_csv(
    "../results/comparacion/comparacion_final_verificada.csv",
    index=False,
    encoding="utf-8-sig"
)

print("COMPARACIÓN EXPERIMENTAL VERIFICADA:")
print(tabla_final.round(6).to_string(index=False))


COMPARACIÓN EXPERIMENTAL VERIFICADA:
    modelo  n_original  silhouette_original  davies_original  n_comun  silhouette_comun  davies_comun  cobertura_total
   K-Means         244             0.493255         0.898488      201          0.586421      0.625052          1.00000
 K-Medoids         244             0.492114         0.897678      201          0.581485      0.640508          1.00000
Jerárquico         244             0.497297         0.815482      201          0.593727      0.598189          1.00000
   HDBSCAN         201             0.593727         0.598189      201          0.593727      0.598189          0.82377
